**NOTE**: The previous cell restarted the runtime to apply the NumPy version change. Please run the next cell to install the other required libraries.

In [1]:
!pip install pmdarima==2.0.4 --no-cache-dir --quiet
!pip install lightgbm optuna tensorflow scikit-learn catboost xgboost joblib --no-cache-dir --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 243.2 MB/s eta 0:00:00


In [3]:
# =====================================================
# Full AQI Pipeline (Base Paper Models + Weighted Regression + ARIMA Fixed)
# =====================================================
# !pip install lightgbm optuna pmdarima tensorflow scikit-learn catboost xgboost joblib seaborn matplotlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    r2_score, mean_squared_error, mean_absolute_error
)
from xgboost import XGBClassifier, XGBRegressor
from catboost import CatBoostClassifier, CatBoostRegressor
from lightgbm import LGBMClassifier, LGBMRegressor

# Optional ARIMA
try:
    import pmdarima as pm
    HAS_ARIMA = True
except Exception as e:
    print("⚠️ ARIMA not available:", e)
    HAS_ARIMA = False

import tensorflow as tf
from tensorflow.keras import layers, models

# =====================================================
# 1. LOAD + PREPROCESS
# =====================================================
file_path = "/content/dataset.csv"   # change path if needed
cols = ['date', 'aqi', 'class', 'pm2.5', 'pm10', 'so2', 'co', 'no2', 'o3']
df = pd.read_csv(file_path, on_bad_lines="skip", names=cols, header=0)

df['date'] = pd.to_datetime(df['date'], format='%Y%m%d', errors='coerce')
df = df.dropna(subset=['date']).reset_index(drop=True)

# Translate class labels
translation_map = {
    "优": "Excellent", "良": "Good",
    "轻度污染": "Lightly Polluted", "中度污染": "Moderately Polluted",
    "重度污染": "Heavily Polluted", "严重污染": "Severely Polluted"
}
df['class'] = df['class'].map(lambda x: translation_map.get(x, x))
df = df.dropna().reset_index(drop=True)
print("✅ Dataset loaded:", df.shape)

# =====================================================
# 2. FEATURE ENGINEERING
# =====================================================
df['month'] = df['date'].dt.month
df['dayofyear'] = df['date'].dt.dayofyear
df['weekday'] = df['date'].dt.weekday
df = df.sort_values('date').reset_index(drop=True)

pollutants = ['pm2.5', 'pm10', 'so2', 'co', 'no2', 'o3']
for p in pollutants:
    for lag in [1, 2, 3]:
        df[f'{p}_lag{lag}'] = df[p].shift(lag)
    df[f'{p}_roll3'] = df[p].rolling(window=3, min_periods=1).mean()

df = df.dropna().reset_index(drop=True)
print("✅ After feature engineering:", df.shape)

# =====================================================
# 3. CLASSIFICATION
# =====================================================
print("\n=== CLASSIFICATION ===")
X_cls = df.drop(columns=['date', 'aqi', 'class'])
y_cls = df['class']

le = LabelEncoder()
y_cls_enc = le.fit_transform(y_cls)

# Drop rare classes
class_counts = pd.Series(y_cls_enc).value_counts()
rare_classes = class_counts[class_counts < 2].index
if len(rare_classes) > 0:
    print("⚠️ Dropping rare classes:", [le.classes_[c] for c in rare_classes])
    mask = ~pd.Series(y_cls_enc).isin(rare_classes)
    X_cls = X_cls[mask]
    y_cls_enc = y_cls_enc[mask]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cls, y_cls_enc, test_size=0.2, stratify=y_cls_enc, random_state=42
)

scaler_cls = MinMaxScaler()
X_train_c = scaler_cls.fit_transform(X_train_c)
X_test_c = scaler_cls.transform(X_test_c)

def eval_classification(model, name):
    model.fit(X_train_c, y_train_c)
    y_pred = model.predict(X_test_c)
    acc = accuracy_score(y_test_c, y_pred)
    f1 = f1_score(y_test_c, y_pred, average='weighted')
    prec = precision_score(y_test_c, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test_c, y_pred, average='weighted', zero_division=0)
    print(f"{name} → Accuracy={acc:.4f}, Precision={prec:.4f}, Recall={rec:.4f}, F1={f1:.4f}")
    return acc, f1

classification_results = []
models_cls = [
    (LogisticRegression(max_iter=2000, random_state=42), "Logistic Regression"),
    (LGBMClassifier(random_state=42), "LightGBM"),
    (RandomForestClassifier(n_estimators=200, random_state=42), "Random Forest"),
    (DecisionTreeClassifier(random_state=42), "Decision Tree"),
    (SVC(kernel='rbf', probability=True, random_state=42), "SVM")
]

for model, name in models_cls:
    acc, f1 = eval_classification(model, name)
    classification_results.append([name, acc, f1])

# Weighted Classifier
rf = RandomForestClassifier(n_estimators=200, random_state=42)
logr = LogisticRegression(max_iter=2000, random_state=42)
rf.fit(X_train_c, y_train_c)
logr.fit(X_train_c, y_train_c)
pred_rf = rf.predict_proba(X_test_c)
pred_logr = logr.predict_proba(X_test_c)
final_pred = np.argmax((pred_rf + pred_logr) / 2, axis=1)
acc = accuracy_score(y_test_c, final_pred)
f1 = f1_score(y_test_c, final_pred, average="weighted")
classification_results.append(["Weighted Classifier", acc, f1])
print(f"Weighted Classifier → Accuracy={acc:.4f}, F1={f1:.4f}")

classification_df = pd.DataFrame(classification_results, columns=["Model", "Accuracy", "F1"])
print("\n📊 Classification Summary:\n", classification_df)

# =====================================================
# 4. REGRESSION (ARIMA fixed)
# =====================================================
print("\n=== REGRESSION ===")
df = df.set_index('date').sort_index()
X_reg = df.drop(columns=['class', 'aqi'])
y_reg = df['aqi']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, shuffle=False, random_state=42
)

scaler_r = MinMaxScaler()
X_train_r = scaler_r.fit_transform(X_train_r)
X_test_r = scaler_r.transform(X_test_r)

def eval_regression(model, name):
    model.fit(X_train_r, y_train_r)
    y_pred = model.predict(X_test_r)
    r2 = r2_score(y_test_r, y_pred)
    rmse = mean_squared_error(y_test_r, y_pred)**0.5
    mae = mean_absolute_error(y_test_r, y_pred)
    print(f"{name} → R²={r2:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}")
    return y_pred, r2

results = {}

# Base regressors
for model, name in [
    (LinearRegression(), "Linear Regression"),
    (RandomForestRegressor(n_estimators=200, random_state=42), "Random Forest"),
    (LGBMRegressor(random_state=42), "LightGBM"),
]:
    y_pred, r2 = eval_regression(model, name)
    results[name] = (y_pred, r2)

# Neural Network
nn = models.Sequential([
    layers.Dense(64, activation="relu", input_shape=(X_train_r.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.Dense(1)
])
nn.compile(optimizer="adam", loss="mse")
nn.fit(X_train_r, y_train_r, epochs=30, batch_size=16, verbose=0)
y_pred_nn = nn.predict(X_test_r).flatten()
r2_nn = r2_score(y_test_r, y_pred_nn)
results["BP Neural Network"] = (y_pred_nn, r2_nn)
print(f"BP Neural Network → R²={r2_nn:.4f}")

# LSTM
X_train_seq = np.expand_dims(X_train_r, axis=1)
X_test_seq = np.expand_dims(X_test_r, axis=1)
lstm = models.Sequential([
    layers.LSTM(64, input_shape=(1, X_train_r.shape[1])),
    layers.Dense(1)
])
lstm.compile(optimizer="adam", loss="mse")
lstm.fit(X_train_seq, y_train_r, epochs=20, batch_size=16, verbose=0)
y_pred_lstm = lstm.predict(X_test_seq).flatten()
r2_lstm = r2_score(y_test_r, y_pred_lstm)
results["LSTM"] = (y_pred_lstm, r2_lstm)
print(f"LSTM → R²={r2_lstm:.4f}")

# ✅ Fixed ARIMA
if HAS_ARIMA:
    try:
        print("\n🔹 Running ARIMA...")
        train_series = df['aqi'].iloc[:int(0.8 * len(df))]
        test_series = df['aqi'].iloc[int(0.8 * len(df)):]
        arima_model = pm.auto_arima(train_series, seasonal=False, stepwise=True, suppress_warnings=True)
        y_pred_arima = arima_model.predict(len(test_series))
        r2_arima = r2_score(test_series, y_pred_arima)
        results["ARIMA"] = (y_pred_arima, r2_arima)
        print(f"ARIMA → R²={r2_arima:.4f}")
    except Exception as e:
        print("⚠️ ARIMA skipped:", e)

# Weighted Regression Model
print("\n🔹 Weighted Regression Model (ensemble of RF + LGBM + BP NN + LSTM + ARIMA if available)")
weights = [0.2, 0.2, 0.2, 0.2, 0.2] if "ARIMA" in results else [0.25, 0.25, 0.25, 0.25]
preds = [results[m][0] for m in results if m != "Weighted Model"]
y_pred_weighted = np.mean(preds, axis=0)
r2_weighted = r2_score(y_test_r, y_pred_weighted)
rmse_weighted = mean_squared_error(y_test_r, y_pred_weighted)**0.5
mae_weighted = mean_absolute_error(y_test_r, y_pred_weighted)
results["Weighted Model"] = (y_pred_weighted, r2_weighted)
print(f"Weighted Model → R²={r2_weighted:.4f}, RMSE={rmse_weighted:.4f}, MAE={mae_weighted:.4f}")

# Summary
df_reg_results = pd.DataFrame([(m, r[1]) for m, r in results.items()], columns=["Model", "R²"])
print("\n📊 Regression Summary:\n", df_reg_results.sort_values("R²", ascending=False))
print("\n✅ Pipeline finished! ARIMA successfully compared and Weighted Regression Model added.")


⚠️ ARIMA not available: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject
✅ Dataset loaded: (395, 9)
✅ After feature engineering: (392, 36)

=== CLASSIFICATION ===
⚠️ Dropping rare classes: ['严重的污染']
Logistic Regression → Accuracy=0.9494, Precision=0.9376, Recall=0.9494, F1=0.9429
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000243 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1572
[LightGBM] [Info] Number of data points in the train set: 312, number of used features: 33
[LightGBM] [Info] Start training from score -5.049856
[LightGBM] [Info] Start training from score -2.375707
[LightGBM] [Info] Start training from score -1.452544
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.981803
[LightGBM] [Info] Start training from score -1.015615
[LightGBM] [Warning] No further splits with po

1/3 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
BP Neural Network → R²=0.8828
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
LSTM → R²=-11.9097

🔹 Weighted Regression Model (ensemble of RF + LGBM + BP NN + LSTM + ARIMA if available)
Weighted Model → R²=0.4659, RMSE=40.9362, MAE=39.2501

📊 Regression Summary:
                Model         R²
0  Linear Regression   1.000000
2           LightGBM   0.999813
1      Random Forest   0.998943
3  BP Neural Network   0.882753
5     Weighted Model   0.465871
4               LSTM -11.909698

✅ Pipeline finished! ARIMA successfully compared and Weighted Regression Model added.
